In [ ]:
##Error Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
FILE_PATH = "myv_ft_20260815_103354_annotated.conllu"

In [ ]:
def parse_conllu(path):
    sentences = []
    current = []
    skip = 0

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                if current:
                    sentences.append(current)
                    current = []
                continue
            if line.startswith("#"):
                continue
            part = line.split("\t")
            if len(part) < 3:
                part = line.split()
            if len(part) < 3:
                skip += 1 
                continue

            word, gold, pred = part[0], part[1], part[2]
            current.append((word, gold, pred))
    if current:
        sentences.append(current)
    if skip:
        print(f"Skipped {skip} lines")

    rows = []
    for sent_id, sent in enumerate(sentences):
        for pos, (word, gold, pred) in enumerate(sent):
            rows.append((word, gold, pred, sent_id, pos))
    df = pd.DataFrame(rows, columns=["word", "gold", "pred", "sent_id", "position"])
    return df, sentences

df, sentences = parse_conllu(FILE_PATH)
df.head()

In [ ]:
df['correct'] = df['gold'] == df['pred']
overall_acc = df['correct'].mean()
print(f"Overall accuracy: {overall_acc:.4%}  ({df['correct'].sum()}/{len(df)})")

Per-tag Evaluation Metrics

In [ ]:
labels = sorted(set(df['gold']) | set(df['pred']))

report = classification_report(df['gold'], df['pred'], labels=labels,
                                zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df

Confusion Matrices

In [ ]:
#Raw
cm = confusion_matrix(df['gold'], df['pred'], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = 'gold \\ pred'
cm_df

In [ ]:
#Percentage
row_sums = cm_df.sum(axis=1)
cm_pct_df = cm_df.div(row_sums, axis=0).fillna(0) * 100
cm_pct_df = cm_pct_df.round(2)
cm_pct_df

In [ ]:
#Heatmap
fig, ax = plt.subplots(figsize=(max(6, len(labels)*0.7), max(5, len(labels)*0.6)))
im = ax.imshow(cm_pct_df.values, cmap='Blues')

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted tag')
ax.set_ylabel('Gold tag')
ax.set_title('Confusion matrix (% of gold tags)')

for i in range(len(labels)):
    for j in range(len(labels)):
        val = cm_pct_df.values[i, j]
        if val > 0:
            ax.text(j, i, f"{val:.0f}", ha='center', va='center',
                    color='white' if val > 50 else 'black', fontsize=8)

fig.colorbar(im, ax=ax, label='%')
plt.tight_layout()
plt.show()

Mistagging Summary

In [ ]:
def mistagging_summary(df, min_pct=0.0, top_n=None):

    lines = []
    for gold_tag in sorted(df['gold'].unique()):
        subset = df[df['gold'] == gold_tag]
        total = len(subset)
        acc = (subset['gold'] == subset['pred']).mean()

        lines.append(f"\n~~~ {gold_tag} (n={total}, accuracy={acc:.1%}) ~~~")

        errors = subset[subset['gold'] != subset['pred']]
        if errors.empty:
            lines.append(f"No errors.")
            continue

        counts = errors['pred'].value_counts()
        shown = 0
        for pred_tag, count in counts.items():
            pct = count / total * 100
            if pct < min_pct:
                continue
            lines.append(f"{gold_tag} was mistagged as {pred_tag} "
                          f"{pct:.1f}% of the time ({count}/{total} tokens).")
            shown += 1
            if top_n is not None and shown >= top_n:
                break

    summary = "\n".join(lines)
    print(summary)
    return summary

_ = mistagging_summary(df)

Tag-Pair Error Analysis

In [ ]:
def show_examples(df, gold_tag, pred_tag, n=20):
    sub = df[(df['gold'] == gold_tag) & (df['pred'] == pred_tag)]
    print(f"{len(sub)} example(s) of gold={gold_tag} but pred={pred_tag}:")
    return sub[['word', 'gold', 'pred']].head(n)

show_examples(df, 'ADJ', 'NOUN', n=50)